In [1]:
import numpy as np

np.random.seed(42)
num_assets = 5

expected_returns = np.array([0.12, 0.15, 0.08, 0.18, 0.10])

random_matrix = np.random.rand(num_assets, num_assets)
cov_matrix = np.dot(random_matrix, random_matrix.T) / 100
risk_free_rate = 0.03


def portfolio_fitness(weights):
    weights = np.array(weights)
    weights = np.maximum(weights, 0)
    if np.sum(weights) == 0:
        return 0
    weights = weights / np.sum(weights)

    port_return = np.dot(weights, expected_returns)

    port_volatility = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))

    sharpe_ratio = (port_return - risk_free_rate) / port_volatility
    return -sharpe_ratio


def optimize_portfolio_pso(
    num_particles=30, max_iter=100, w=0.5, c1=1.5, c2=1.5
):
    dim = num_assets

    positions = np.random.rand(num_particles, dim)
    positions = positions / np.sum(positions, axis=1, keepdims=True)

    velocities = np.random.uniform(-0.1, 0.1, (num_particles, dim))

    p_best_positions = positions.copy()
    p_best_scores = np.array(
        [portfolio_fitness(p) for p in positions]
    )

    g_best_idx = np.argmin(p_best_scores)
    g_best_position = p_best_positions[g_best_idx].copy()
    g_best_score = p_best_scores[g_best_idx]

    for iteration in range(max_iter):
        for i in range(num_particles):
            r1, r2 = np.random.rand(), np.random.rand()

            velocities[i] = (
                w * velocities[i]
                + c1 * r1 * (p_best_positions[i] - positions[i])
                + c2 * r2 * (g_best_position - positions[i])
            )

            positions[i] += velocities[i]

            positions[i] = np.maximum(positions[i], 0)
            if np.sum(positions[i]) > 0:
                positions[i] = positions[i] / np.sum(positions[i])

            score = portfolio_fitness(positions[i])

            if score < p_best_scores[i]:
                p_best_scores[i] = score
                p_best_positions[i] = positions[i].copy()

                if score < g_best_score:
                    g_best_score = score
                    g_best_position = positions[i].copy()

    g_best_position = g_best_position / np.sum(g_best_position)
    return g_best_position, -g_best_score


best_weights, best_sharpe = optimize_portfolio_pso()

print("--- Optimal Portfolio Weights ---")
for i, w in enumerate(best_weights):
    print(f"Asset {i+1}: {w * 100:.2f}%")

print(f"\nMaximized Sharpe Ratio: {best_sharpe:.4f}")

--- Optimal Portfolio Weights ---
Asset 1: 0.00%
Asset 2: 0.00%
Asset 3: 0.00%
Asset 4: 100.00%
Asset 5: 0.00%

Maximized Sharpe Ratio: 1.8285
